# Reproduce: Neighbor-Interpolated Point Cloud Densification for 3DGS

**Run this notebook top-to-bottom on Google Colab (T4 GPU, free tier).**

Steps:
1. Mount Google Drive (to survive session resets)
2. Clone 3DGS repo + build CUDA extensions
3. Download dataset
4. Densify the point cloud
5. Train baseline + densified (7000 iterations each)
6. Evaluate and compare
7. Qualitative figures

> **Important:** Select `Runtime > Change runtime type > T4 GPU` before running.

## Step 1 — Mount Google Drive
Keeps checkpoints safe across session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/thesis/output', exist_ok=True)
print('Drive mounted. Output will be saved to /content/drive/MyDrive/thesis/output/')

## Step 2 — Clone 3DGS and build CUDA extensions

In [ ]:
%cd /content
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd /content/gaussian-splatting

import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'  # T4 compute capability
!pip install submodules/diff-gaussian-rasterization -q
!pip install submodules/simple-knn -q
!pip install lpipsPyTorch -q

# CRITICAL: import torch FIRST before importing the extensions
import torch
from simple_knn._C import distCUDA2
from diff_gaussian_rasterization import GaussianRasterizationSettings
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print('Both CUDA extensions OK')

## Step 3 — Download the dataset (Tanks & Temples)

In [ ]:
%cd /content/gaussian-splatting
!wget https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -q --show-progress
!unzip -q tandt_db.zip -d data
print('Files in data/tandt/truck/sparse/0:')
!ls data/tandt/truck/sparse/0

## Step 4 — Densify the point cloud
Copy the truck scene to a second folder, then run the densification script on the copy.

In [ ]:
!pip install plyfile scipy -q
!cp -r data/tandt/truck data/tandt/truck_densified

In [ ]:
%%writefile /content/densify_init_pointcloud.py
import argparse, os, struct
import numpy as np
from scipy.spatial import cKDTree
from plyfile import PlyData, PlyElement

def load_points3D_bin(path):
    xyz, rgb = [], []
    with open(path, 'rb') as f:
        num_points = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num_points):
            f.read(8)
            x, y, z = struct.unpack('<ddd', f.read(24))
            r, g, b = struct.unpack('<BBB', f.read(3))
            f.read(8)
            track_length = struct.unpack('<Q', f.read(8))[0]
            f.read(8 * track_length)
            xyz.append((x, y, z)); rgb.append((r, g, b))
    return np.array(xyz, dtype=np.float32), np.array(rgb, dtype=np.uint8)

def load_ply(path):
    v = PlyData.read(path)['vertex']
    return np.stack([v['x'],v['y'],v['z']],1).astype(np.float32), np.stack([v['red'],v['green'],v['blue']],1).astype(np.uint8)

def load_points3D(path):
    ext = os.path.splitext(path)[1].lower()
    return load_points3D_bin(path) if ext=='.bin' else load_ply(path)

def save_ply(path, xyz, rgb):
    normals = np.zeros_like(xyz)
    dtype = [('x','f4'),('y','f4'),('z','f4'),('nx','f4'),('ny','f4'),('nz','f4'),('red','u1'),('green','u1'),('blue','u1')]
    elements = np.empty(xyz.shape[0], dtype=dtype)
    elements[:] = list(map(tuple, np.concatenate([xyz,normals,rgb.astype(np.float32)],1)))
    PlyData([PlyElement.describe(elements,'vertex')],text=False).write(path)

def densify(xyz, rgb, k=6, factor=3.0, max_new=1):
    tree = cKDTree(xyz)
    dists, idxs = tree.query(xyz, k=k+1)
    threshold = factor * np.median(dists[:,1])
    print(f'Threshold: {threshold:.5f} ({factor}x median nn dist)')
    new_xyz, new_rgb, seen = [], [], set()
    for i in range(len(xyz)):
        inserted = 0
        for rank in range(1, k+1):
            if inserted >= max_new: break
            j = int(idxs[i,rank])
            if dists[i,rank] <= threshold: continue
            pair = (min(i,j),max(i,j))
            if pair in seen: continue
            seen.add(pair)
            new_xyz.append((xyz[i]+xyz[j])/2)
            new_rgb.append(((rgb[i].astype(np.float32)+rgb[j].astype(np.float32))/2).astype(np.uint8))
            inserted += 1
    if new_xyz:
        return np.concatenate([xyz,np.stack(new_xyz)]), np.concatenate([rgb,np.stack(new_rgb)])
    return xyz, rgb

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--input', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--k', type=int, default=6)
    p.add_argument('--factor', type=float, default=3.0)
    p.add_argument('--max_new_per_point', type=int, default=1)
    args = p.parse_args()
    xyz, rgb = load_points3D(args.input)
    print(f'Original: {len(xyz)} points')
    xyz, rgb = densify(xyz, rgb, args.k, args.factor, args.max_new_per_point)
    print(f'Densified: {len(xyz)} points')
    save_ply(args.output, xyz, rgb)
    print(f'Saved: {args.output}')

In [ ]:
!python /content/densify_init_pointcloud.py \
    --input data/tandt/truck_densified/sparse/0/points3D.bin \
    --output data/tandt/truck_densified/sparse/0/points3D.ply \
    --k 6 --factor 3.0 --max_new_per_point 1

## Step 5 — Train both conditions
> Stay on this tab while training. Each run takes ~15-20 min on T4.

In [ ]:
# Baseline
!python train.py \
    -s data/tandt/truck \
    -m /content/drive/MyDrive/thesis/output/truck_baseline \
    --eval \
    --iterations 7000 \
    --test_iterations 1000 3000 7000 \
    --save_iterations 1000 3000 7000 \
    --checkpoint_iterations 1000 3000 7000

In [ ]:
# Densified init
!python train.py \
    -s data/tandt/truck_densified \
    -m /content/drive/MyDrive/thesis/output/truck_densified \
    --eval \
    --iterations 7000 \
    --test_iterations 1000 3000 7000 \
    --save_iterations 1000 3000 7000 \
    --checkpoint_iterations 1000 3000 7000

## Step 6 — Render and evaluate

In [ ]:
for condition in ['truck_baseline', 'truck_densified']:
    model_path = f'/content/drive/MyDrive/thesis/output/{condition}'
    for it in [1000, 3000, 7000]:
        os.system(f'python render.py -m {model_path} --iteration {it}')
    os.system(f'python metrics.py -m {model_path}')
    print(f'Done: {condition}')

In [ ]:
%%writefile /content/compare_results.py
import argparse, json

def load(path):
    with open(path) as f: return json.load(f)

def key_to_iter(k):
    d = ''.join(c for c in k if c.isdigit())
    return int(d) if d else -1

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--baseline', required=True)
    p.add_argument('--modified', required=True)
    args = p.parse_args()
    b, m = load(args.baseline), load(args.modified)
    keys = sorted((k for k in b if k in m), key=key_to_iter)
    def get(d, name):
        for c in (name, name.upper(), name.lower(), name.capitalize()):
            if c in d: return d[c]
    print(f'{"Iter":>8} | {"PSNR base":>10} {"PSNR mod":>10} {"Delta":>7} | {"SSIM base":>10} {"SSIM mod":>9} {"Delta":>7} | {"LPIPS base":>11} {"LPIPS mod":>10} {"Delta":>7}')
    print('-'*100)
    for k in keys:
        it = key_to_iter(k)
        pb,pm = get(b[k],'PSNR'),get(m[k],'PSNR')
        sb,sm = get(b[k],'SSIM'),get(m[k],'SSIM')
        lb,lm = get(b[k],'LPIPS'),get(m[k],'LPIPS')
        print(f'{it:8d} | {pb:10.3f} {pm:10.3f} {pm-pb:+7.3f} | {sb:10.4f} {sm:9.4f} {sm-sb:+7.4f} | {lb:11.4f} {lm:10.4f} {lm-lb:+7.4f}')
    print('\nPositive Delta PSNR/SSIM = modified better. Negative Delta LPIPS = modified better.')

In [ ]:
!python /content/compare_results.py \
    --baseline /content/drive/MyDrive/thesis/output/truck_baseline/results.json \
    --modified /content/drive/MyDrive/thesis/output/truck_densified/results.json

## Step 7 — Qualitative figures and per-view analysis

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

renders_base = '/content/drive/MyDrive/thesis/output/truck_baseline/test/ours_7000/renders'
renders_dense = '/content/drive/MyDrive/thesis/output/truck_densified/test/ours_7000/renders'
gt_dir = '/content/drive/MyDrive/thesis/output/truck_baseline/test/ours_7000/gt'

# Per-view L1 analysis
import os
view_ids = sorted(os.listdir(gt_dir))
results = []
for vid in view_ids:
    gt = np.array(Image.open(f'{gt_dir}/{vid}')).astype(float)
    base = np.array(Image.open(f'{renders_base}/{vid}')).astype(float)
    dense = np.array(Image.open(f'{renders_dense}/{vid}')).astype(float)
    results.append((vid, np.abs(gt-base).mean(), np.abs(gt-dense).mean()))

from scipy import stats
deltas = [r[2]-r[1] for r in results]
wins = sum(1 for d in deltas if d < 0)
_, w_p = stats.wilcoxon(deltas)
print(f'Densified better in {wins}/{len(results)} views')
print(f'Mean delta L1: {np.mean(deltas):+.4f}, Wilcoxon p={w_p:.4f}')

# Best / worst view comparison figure
best = sorted(results, key=lambda r: r[2]-r[1])[0][0]
worst = sorted(results, key=lambda r: r[2]-r[1])[-1][0]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for row, (vid, label) in enumerate([(best,'Best case for densified'), (worst,'Worst case for densified')]):
    gt = Image.open(f'{gt_dir}/{vid}')
    base = Image.open(f'{renders_base}/{vid}')
    dense = Image.open(f'{renders_dense}/{vid}')
    for col, (img, title) in enumerate(zip([gt,base,dense],['Ground Truth','Baseline','Densified Init'])):
        axes[row,col].imshow(img)
        axes[row,col].set_title(f'{title}\n{label} ({vid})')
        axes[row,col].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/thesis/qualitative_best_worst.png', dpi=150, bbox_inches='tight')
plt.show()

# Error maps
gt = np.array(Image.open(f'{gt_dir}/{best}')).astype(float)
base = np.array(Image.open(f'{renders_base}/{best}')).astype(float)
dense = np.array(Image.open(f'{renders_dense}/{best}')).astype(float)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(np.abs(gt-base).mean(-1), cmap='hot', vmin=0, vmax=30)
axes[0].set_title(f'Baseline error (view {best})')
axes[1].imshow(np.abs(gt-dense).mean(-1), cmap='hot', vmin=0, vmax=30)
axes[1].set_title(f'Densified error (view {best})')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/thesis/error_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures saved to Drive.')